In [ ]:
from pathlib import Path
import cv2
import numpy as np
from PIL import Image

photo_path = Path("../tests/slides/real/olympia_phone_2000.jpg")
photo = cv2.imread(str(photo_path))                  # фото → массив чисел; порядок цветов B, G, R
print("фото:", photo.shape, "· тип:", photo.dtype)

gray = cv2.cvtColor(photo, cv2.COLOR_BGR2GRAY)       # три цвета → одна яркость
print("серое:", gray.shape, "· яркость от", gray.min(), "до", gray.max())
Image.fromarray(gray)                                # показать массив как картинку

In [ ]:
blur = cv2.GaussianBlur(gray, (5, 5), 0)                  # убрать зерно и муар: края — по крупным границам
edges = cv2.Canny(blur, 50, 150)                          # 255 — резкий перепад яркости, 0 — нет
edges = cv2.dilate(edges, np.ones((3, 3), np.uint8))      # утолщить края — сомкнуть разрывы рамки

contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print("контуров:", len(contours), "· доля точек-краёв:", round((edges > 0).mean() * 100, 1), "%")
Image.fromarray(edges)


In [ ]:
h, w = gray.shape
candidates = sorted(contours, key=cv2.contourArea, reverse=True)[:5]   # 5 самых больших контуров

frame = None                                                  # пока «не найдено»
for contour in candidates:
    perimeter = cv2.arcLength(contour, True)
    corners = cv2.approxPolyDP(contour, 0.02 * perimeter, True)   # спрямить дрожь до немногих углов
    share = cv2.contourArea(corners) / (w * h)
    print("углов:", len(corners), "· доля кадра:", round(share, 2))
    if len(corners) == 4 and share > 0.15:
        frame = corners.reshape(4, 2)                         # 4 угла по (x, y)
        break

print("рамка слайда:", frame.tolist() if frame is not None else "не найдена")

preview = photo.copy()                                        # копия — рисуем на ней, оригинал не трогаем
if frame is not None:
    cv2.polylines(preview, [frame], True, (0, 255, 0), 8)     # зелёная рамка толщиной 8
Image.fromarray(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))

In [ ]:
def order_corners(pts):
    """4 угла в любом порядке → верхний левый, верхний правый, нижний правый, нижний левый."""
    pts = pts.astype("float32")
    s = pts.sum(axis=1)                  # x + y: меньше всех у верхнего левого, больше всех у нижнего правого
    d = pts[:, 1] - pts[:, 0]            # y − x: меньше всех у верхнего правого, больше всех у нижнего левого
    return np.array([pts[s.argmin()], pts[d.argmin()], pts[s.argmax()], pts[d.argmax()]], dtype="float32")


src = order_corners(frame)
tl, tr, br, bl = src                                          # верх-лево, верх-право, низ-право, низ-лево
out_w = int(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)))   # ширина — по длинной горизонтальной стороне
out_h = int(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr)))   # высота — по длинной вертикальной
dst = np.array([[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1], [0, out_h - 1]], dtype="float32")

matrix = cv2.getPerspectiveTransform(src, dst)                # правило: трапеция на фото → прямоугольник
slide_img = cv2.warpPerspective(photo, matrix, (out_w, out_h))
print("углы по порядку:", src.astype(int).tolist())
print("выпрямленный слайд:", out_w, "×", out_h, "· отношение сторон:", round(out_w / out_h, 2))
Image.fromarray(cv2.cvtColor(slide_img, cv2.COLOR_BGR2RGB))

In [ ]:
def quality(img):
    """Три числа качества выпрямленного слайда: ширина, резкость, пересвет."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    width = gray.shape[1]
    same_size = cv2.resize(gray, (1280, round(gray.shape[0] * 1280 / width)))   # к общей ширине — для сравнения
    sharpness = cv2.Laplacian(same_size, cv2.CV_64F).var()                      # резкость: разброс перепадов
    overexposed = (gray >= 250).mean() * 100                                    # доля почти белых точек, %
    return {"ширина": width, "резкость": round(sharpness), "пересвет, %": round(float(overexposed), 1)}


print("фото, выпрямленный:", quality(slide_img))
screen = cv2.imread(str(sorted(Path("../tests/slides/real").glob("*.png"))[0]))
print("снимок экрана:     ", quality(screen))
blurred = cv2.GaussianBlur(slide_img, (9, 9), 0)
print("фото, размыто:     ", quality(blurred))

In [ ]:
import hashlib
from collections import Counter

SLIDE_RATIO = (1.25, 1.95)     # выпрямленная рамка: от 4:3 (1,33) до 16:9 (1,78) с запасом на перекос
WHOLE_RATIO = (1.6, 2.2)       # «весь кадр — уже слайд» только для широких кадров: камера телефона снимает 4:3
MIN_SHARE = 0.15               # рамка должна занимать не меньше 15 % кадра


def find_frame(photo):
    """Фото → 4 угла самого большого четырёхугольника (≥ 15 % кадра) или None."""
    gray = cv2.cvtColor(photo, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 50, 150)
    edges = cv2.dilate(edges, np.ones((3, 3), np.uint8))
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    h, w = gray.shape
    for contour in sorted(contours, key=cv2.contourArea, reverse=True)[:5]:
        corners = cv2.approxPolyDP(contour, 0.02 * cv2.arcLength(contour, True), True)
        if len(corners) == 4 and cv2.contourArea(corners) / (w * h) > MIN_SHARE:
            return corners.reshape(4, 2)
    return None


def straighten(photo, frame):
    """Фото и 4 угла → выпрямленный прямоугольник."""
    src = order_corners(frame)
    tl, tr, br, bl = src
    out_w = int(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)))
    out_h = int(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr)))
    dst = np.array([[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1], [0, out_h - 1]], dtype="float32")
    return cv2.warpPerspective(photo, cv2.getPerspectiveTransform(src, dst), (out_w, out_h))


def ratio_in(img, limits):
    """Лежит ли отношение ширины к высоте картинки в пределах limits = (от, до)."""
    ratio = img.shape[1] / img.shape[0]
    return limits[0] <= ratio <= limits[1]


def capture(photo):
    """Фото → (картинка слайда или None, исход): «рамка», «весь кадр» или «не найдено»."""
    frame = find_frame(photo)
    if frame is not None:
        slide = straighten(photo, frame)
        if ratio_in(slide, SLIDE_RATIO):
            return slide, "рамка"
    if ratio_in(photo, WHOLE_RATIO):
        return photo, "весь кадр"
    return None, "не найдено"

In [ ]:
results = []
seen = set()
for path in sorted(Path("../tests/slides/real").iterdir()):
    if path.suffix.lower() not in (".jpg", ".png"):
        continue
    digest = hashlib.md5(path.read_bytes()).hexdigest()   # «отпечаток» содержимого файла
    if digest in seen:
        continue                                          # такой файл уже был — дубль, пропускаем
    seen.add(digest)
    photo = cv2.imread(str(path))
    slide, outcome = capture(photo)
    results.append((path.name, outcome))

print("файлов (без дублей):", len(results), "·", dict(Counter(outcome for name, outcome in results)))
for name, outcome in results:
    if outcome == "не найдено":
        print("   не найдено:", name)

In [ ]:
import json
import re
from rapidocr import RapidOCR

gold = json.loads(Path("../tests/gold/olympia_2021.json").read_text(encoding="utf-8"))
gold_values = []
for code, fields in gold["endpoints"].items():
    for field, value in fields.items():
        gold_values.append(value)
print("значений в эталоне:", len(gold_values))

ocr_engine = RapidOCR()


def ocr_found(img, wanted):
    """Сколько значений из wanted OCR находит на картинке: число — среди чисел, строка — в тексте."""
    result = ocr_engine(Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))
    raw = " ".join(result.txts)                          # строки OCR через пробел — числа не слипаются
    text = raw.lower().replace(" ", "")                  # для поиска строк вроде «<0.001»
    numbers = set()
    for token in re.findall(r"\d+(?:[.,]\d+)?", raw):
        numbers.add(float(token.replace(",", ".")))
    found = 0
    for value in wanted:
        if isinstance(value, str):
            found += value.lower().replace(" ", "") in text
        else:
            found += float(value) in numbers
    return found

In [ ]:
import time

photo = cv2.imread("../tests/slides/real/IMG_20260919_124700.jpg")
big, outcome = capture(photo)
print("исход:", outcome, "· слайд:", big.shape[1], "×", big.shape[0])

for width in [3600, 2800, 2000, 1600, 1280, 1000, 800, 600]:
    small = cv2.resize(big, (width, round(big.shape[0] * width / big.shape[1])), interpolation=cv2.INTER_AREA)
    t0 = time.perf_counter()
    found = ocr_found(small, gold_values)
    print(f"ширина {width:5} · OCR нашёл {found:2} из 22 · резкость {quality(small)['резкость']:4} · {time.perf_counter() - t0:.1f} с")